# Purpose

The purpose of this notebook is to perform customer-level feature engineering and data exploration using the previously prepared silver-level dataset.

## Import

In [14]:
import pandas as pd
from src.data_vis_plotly import *

## Dataset

Let's load our silver layer

In [15]:
# Load Silver data
df = pd.read_csv("dataset/silver/silver_merged_data.csv", parse_dates=['invoice_date', 'due_date', 'settled_date'])


In [16]:
df

,customer_account,invoice_voucher,invoice_date,due_date,amount,amount_settled,settled_date,balance,sme_mbs,segment_id,...,missing_experian_credit_score,missing_segment_id,invoice_month,days_to_pay,aged_debt,aged_balance,is_unpaid,days_overdue,is_late,payment_status
0,880000000010,IV00000097,2023-05-02,2023-05-16,52.26,52.26,2023-05-16,0.00,SME,FARMING_AGRICULTURE,...,False,False,2023-05-01,14.0,False,0.00,False,0,False,On Time
1,880000000010,IV00006738,2023-07-24,2023-08-07,48.25,48.25,2023-08-07,0.00,SME,FARMING_AGRICULTURE,...,False,False,2023-07-01,14.0,False,0.00,False,0,False,On Time
2,880000000010,IV00047259,2023-10-31,2023-11-14,57.26,57.26,2023-12-01,0.00,SME,FARMING_AGRICULTURE,...,False,False,2023-10-01,31.0,False,0.00,False,0,True,Late
3,880000000010,IV00314258,2024-02-15,2024-02-29,1565.57,1565.57,2024-02-29,0.00,SME,FARMING_AGRICULTURE,...,False,False,2024-02-01,14.0,False,0.00,False,0,False,On Time
4,880000000010,IV00801038,2024-04-28,2024-05-12,1295.81,1295.81,2024-05-13,0.00,SME,FARMING_AGRICULTURE,...,False,False,2024-04-01,15.0,False,0.00,False,0,True,Late
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177215,880000240410,IV02651398,2025-03-19,2025-04-02,5604.36,0.00,NaT,5604.36,SME,REAL_ESTATE,...,True,False,2025-03-01,NaN,True,5604.36,True,85,False,Unpaid
177216,880000240410,IV02665601,2025-03-21,2025-04-04,719.04,0.00,NaT,719.04,SME,REAL_ESTATE,...,True,False,2025-03-01,NaN,True,719.04,True,83,False,Unpaid
177217,880000243959,IV02684842,2025-03-28,2025-04-11,100.16,0.00,NaT,100.16,SME,UNCLASSIFIED,...,True,False,2025-03-01,NaN,True,100.16,True,76,False,Unpaid
177218,880000243959,IV02687134,2025-03-29,2025-04-12,144.48,0.00,NaT,144.48,SME,UNCLASSIFIED,...,True,False,2025-03-01,NaN,True,144.48,True,75,False,Unpaid



## **Customer-Level Aggregation and Feature Engineering**

In the previous notebook, the focus was primarily on row-level feature engineering. In this section, we will shift to customer-level feature engineering, aggregating transactional and behavioral data to generate meaningful, customer-specific features.

Customer-level aggregations

In [17]:
agg_df = df.groupby('customer_account').agg(

    # Total number of invoices issued to this customer
    total_invoices=('invoice_voucher', 'count'),

    # Total invoiced amount
    total_amount=('amount', 'sum'),

    # Total amount actually paid (settled)
    total_settled=('amount_settled', 'sum'),

    # Current outstanding balance across all invoices
    total_balance=('balance', 'sum'),

    # Average and median time (in days) it took to pay invoices
    avg_days_to_pay=('days_to_pay', 'mean'),
    median_days_to_pay=('days_to_pay', 'median'),

    # % of invoices paid after the due date
    late_payment_ratio=('is_late', 'mean'),

    # % of invoices still unpaid
    unpaid_ratio=('is_unpaid', 'mean'),

    # Number of aged debts (invoices overdue by 60+ days)
    aged_debt_count=('aged_debt', 'sum'),

    # Total balance amount from only aged debts (precomputed above)
    aged_debt_amount=('aged_balance', 'sum')

).reset_index()

## Average Days to pay invoice

In [18]:
plot_numerical_distribution(agg_df, 'avg_days_to_pay', title=f"Distribution of avg_days_to_pay", palette=[colour[0]], nbins=100)

The chart above shows the distribution of average days to pay per customer. Most customers tend to settle their invoices within 10 to 30 days, which reflects a relatively prompt payment behaviour. However, there is a long tail in the distribution, indicating that some customers take significantly longer to pay, potentially signaling delayed or inconsistent payment patterns.

## Late payment ratio

In [19]:
plot_numerical_distribution(agg_df, 'late_payment_ratio', title=f"Distribution of late_payment_ratio", palette=[colour[1]], nbins=100)

The chart illustrates the distribution of the late payment ratio, which measures the proportion of invoices each customer pays after the due date. A large concentration at zero indicates that many customers consistently pay on time. However, there is a wide spread across the rest of the distribution, suggesting that a notable portion of customers occasionally pay late. Additionally, the spike at one signifies a segment of customers who always pay late.

## Unpaid Ratio

In [20]:
plot_numerical_distribution(agg_df, 'unpaid_ratio', title=f"Distribution of unpaid_ratio", palette=[colour[2]], nbins=100)

The chart above shows the distribution of the unpaid ratio, which represents the proportion of invoices that remain unpaid for each customer. A large concentration at 0 indicates that the majority of customers have settled most or all of their invoices. However, the long right tail and a noticeable spike near 1.0 suggest that there is a segment of customers who leave most or all of their invoices unpaid.

## Unpaid ratio vs Internal Credit Rating


In [21]:
customer_meta = df[['customer_account', 'segment_id', 'internal_credit_rating']].drop_duplicates()
agg_df = agg_df.merge(customer_meta, on='customer_account', how='left')

In [22]:
credit_order = [
    'Great payer', 'Prompt payer', 'Late payer',
    'Avoid payer', 'Bad payer', 'Credit insured'
]

color_map = get_category_color_map(pd.DataFrame({'internal_credit_rating': credit_order}), 'internal_credit_rating')

plot_box_by_group(
    df=agg_df,
    group_col='internal_credit_rating',
    value_col='unpaid_ratio',
    title="Unpaid Ratio by Credit Rating",
    category_palette=color_map
)

In the chart above, we observe that the internal credit ratings generally align with unpaid ratios, supporting the validity of the internal scoring system. Great payers and Prompt payers typically have low unpaid ratios, though both groups show notable outliers - including cases where some customers have unpaid ratios near 1. This suggests that even among top rated segments, a few accounts may not meet expectations. As expected, Late payers, Avoid payers, and Bad payers exhibit a much wider spread in unpaid ratios, indicating a greater tendency for invoices to remain unsettled, reinforcing their classification as higher-risk segments.

## Top 50 Average Age Debt Amount by Segment

In [23]:

color_map = get_category_color_map(agg_df, 'segment_id')


plot_treemap(
    agg_df.sort_values('aged_debt_amount', ascending=False).head(50).reindex(),
    'segment_id',
    value_col='aged_debt_amount',
    title="Top 50 Average Age Debt by Segment",
    category_palette=color_map
)


The chart displays the top 50 industry segments by average aged debt. The Telecommunications segment leads with the highest average aged debt, followed by Veterinary Services and Manufacturing. These segments represent the largest exposure in terms of outstanding older debt, highlighting areas that may require closer credit control or targeted collection efforts.

# Final dataset

In [24]:
customers = df.drop_duplicates(subset='customer_account')[
    ['customer_account', 'internal_credit_rating', 'experian_credit_score',
     'segment_id', 'cust_group', 'sme_mbs', 'payment_method', 'sales_route',
     'segment', 'count_sites', 'tenure_group', 'contracted_annual_volume']
]

# Drop metadata columns 
columns_to_drop = [
    'internal_credit_rating', 'experian_credit_score', 'segment_id',
    'cust_group', 'sme_mbs', 'payment_method', 'sales_route',
    'segment', 'count_sites', 'tenure_group', 'contracted_annual_volume'
]
agg_df_clean = agg_df.drop(columns=[col for col in columns_to_drop if col in agg_df.columns])

# Merge cleanly
gold_df = agg_df_clean.merge(customers, on='customer_account', how='left')



In [25]:
display(gold_df)

,customer_account,total_invoices,total_amount,total_settled,total_balance,avg_days_to_pay,median_days_to_pay,late_payment_ratio,unpaid_ratio,aged_debt_count,...,experian_credit_score,segment_id,cust_group,sme_mbs,payment_method,sales_route,segment,count_sites,tenure_group,contracted_annual_volume
0,880000000010,9,5854.26,5854.26,0.00,14.444444,14.0,0.222222,0.0,0,...,84.0,FARMING_AGRICULTURE,SOLPAR,SME,DD,Direct,Fixed,1,90 days - 6 months,7397
1,880000000013,10,8847.96,8847.96,0.00,11.700000,14.0,0.200000,0.0,0,...,94.0,RETAIL,LTD,SME,DD,Indirect,Fixed,1,1-3 years,6488
2,880000000031,24,2677.62,2677.62,0.00,16.041667,14.0,0.291667,0.0,0,...,44.0,RETAIL,LTD,SME,DD,Direct,VBR,1,<90 days,4581
3,880000000049,24,5689.81,5689.81,0.00,14.541667,14.0,0.333333,0.0,0,...,85.0,HEALTH,LTD,SME,DD,Direct,Fixed,1,6 months - 1 year,8615
4,880000000054,3,15629.39,15629.39,0.00,21.000000,17.0,1.000000,0.0,0,...,83.0,FARMING_AGRICULTURE,SOLPAR,SME,NONDD,Direct,Fixed,1,90 days - 6 months,18373
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9994,880000243901,2,2814.59,0.00,2814.59,NaN,NaN,0.000000,1.0,2,...,99.0,CONSTRUCTION,LTD,SME,NONDD,Direct,VBR,1,6 months - 1 year,30702
9995,880000243959,2,244.64,0.00,244.64,NaN,NaN,0.000000,1.0,2,...,99.0,UNCLASSIFIED,LTD,SME,NONDD,Direct,VBR,1,<90 days,2536
9996,880000244105,2,4284.54,0.00,4284.54,NaN,NaN,0.000000,1.0,2,...,99.0,MANAGEMENT_CONSULTAN,LTD,SME,NONDD,Direct,VBR,1,90 days - 6 months,43748
9997,880000244297,2,3575.92,0.00,3575.92,NaN,NaN,0.000000,1.0,2,...,99.0,PROFESSIONAL_SPECIAL,SOLPAR,SME,NONDD,Direct,VBR,1,6 months - 1 year,18336


In [26]:
gold_df.to_csv("dataset/gold/gold_customer_features.csv", index=False)
print("Saved: dataset/gold/gold_customer_features.csv")


Saved: dataset/gold/gold_customer_features.csv
